# 05e · Do flagged fan-out users share products?
Takes the 25 flagged users with the most distinct products and, for each, their 5 strongest cart/purchase products (by interaction count). Counts how many of those products are shared by two or more of the 25 users and how many connected components the resulting user–product graph has. Deterministic (no community detection). Metrics: `ml/outputs/05e_metrics.json`.

In [1]:
import os, json
import networkx as nx
from neo4j import GraphDatabase
driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
def q(query, **p):
    with driver.session(database="neo4j") as s: return s.run(query, **p).data()

rows = q("""MATCH (u:User {flagged:true})-[:INTERACTED]->(p:Product) WITH u, count(DISTINCT p) AS deg ORDER BY deg DESC, u.userId LIMIT 25
    MATCH (u)-[r:INTERACTED]->(p:Product) WHERE r.type IN ['cart','purchase']
    WITH u, deg, p, sum(r.count) AS w ORDER BY u.userId, w DESC, p.productId
    WITH u, deg, collect(p.productId)[0..5] AS top
    RETURN u.userId AS user, deg, top""")
G = nx.Graph()
for r in rows:
    for p in r["top"]: G.add_edge(("u", r["user"]), ("p", p))
prod_deg = [d for n, d in G.degree() if n[0] == "p"]
metrics = dict(users=len(rows), min_distinct_products=min(r["deg"] for r in rows), max_distinct_products=max(r["deg"] for r in rows),
               users_with_cart_or_purchase=sum(1 for r in rows if r["top"]), products=len(prod_deg),
               products_shared_by_2plus_users=sum(1 for d in prod_deg if d >= 2),
               connected_components=nx.number_connected_components(G))
json.dump(metrics, open("/home/jovyan/work/ml/outputs/05e_metrics.json", "w"), indent=2)
print(metrics)

{'users': 18, 'min_distinct_products': 1142, 'max_distinct_products': 2043, 'users_with_cart_or_purchase': 18, 'products': 69, 'products_shared_by_2plus_users': 0, 'connected_components': 18}


In [2]:
# how many of ALL fan-out users (> 300 distinct products) never carted or bought anything?
v = q("""MATCH (u:User)-[r:INTERACTED]->(p:Product) WITH u, count(DISTINCT p) AS deg, sum(CASE WHEN r.type IN ['cart','purchase'] THEN 1 ELSE 0 END) AS buy_rels
    WHERE deg > 300 RETURN count(*) AS fanout_users, sum(CASE WHEN buy_rels = 0 THEN 1 ELSE 0 END) AS view_only,
    sum(CASE WHEN buy_rels = 0 AND u.flagged THEN 1 ELSE 0 END) AS view_only_flagged, sum(CASE WHEN u.flagged THEN 1 ELSE 0 END) AS flagged""")[0]
metrics["fanout_all"] = v
json.dump(metrics, open("/home/jovyan/work/ml/outputs/05e_metrics.json", "w"), indent=2)
print(v)
driver.close()

{'fanout_users': 3083, 'view_only': 290, 'view_only_flagged': 111, 'flagged': 1987}
